# Kakao Pay PoC — Step 2: Bronze → Silver → Gold (SDP + Expectation)

## 전제

step1.ipynb (Task A tiara + Task B batch) 완료 후 실행.

## 파이프라인 구조

```
kpay_poc.bronze.*  (step1에서 적재된 Delta 테이블)
    └── SDP 파이프라인
          ├── Silver: 정제·표준화·PII 마스킹·Expectation 품질 검증
          └── Gold  : 집계·비즈니스 메트릭 (replaceWhere로 멱등 재실행)
```

## SDP(Lakeflow Spark Declarative Pipelines) 계층 역할

> **참고:** 기존 DLT(Delta Live Tables)가 **Lakeflow Spark Declarative Pipelines**으로 리브랜딩됨.
> - Import: `import dlt` → `from pyspark import pipelines as dp`
> - `@dlt.table` → `@dp.table` (streaming) / `@dp.materialized_view` (batch)
> - `dlt.apply_changes()` → `dp.create_auto_cdc_flow()`
> - 기존 `dlt` 모듈은 하위호환으로 동작하지만 신규 코드는 `pyspark.pipelines` 사용 권장
> - Apache Spark 4.1 오픈소스 declarative pipelines와 동일 API

| 계층 | 역할 | 형태 |
|:---|:---|:---|
| **Bronze** | 원천 그대로의 raw 데이터 (step1 적재) | append-only, 스키마 변동 허용 |
| **Silver** | 정제·타입 통일·PII 마스킹·중복 제거 | Expectation 통과한 행만 |
| **Gold** | 분석/BI용 집계·비즈니스 메트릭 | replaceWhere 멱등 재실행 |

## Expectation 모드

| 데코레이터 | 동작 |
|:---|:---|
| `dp.expect(...)` | 위반 행 기록만, 파이프라인 계속 |
| `dp.expect_or_drop(...)` | 위반 행 제거, 파이프라인 계속 |
| `dp.expect_or_fail(...)` | 위반 시 파이프라인 실패 |
| `dp.expect_all_or_drop(...)` | 여러 조건 한 번에, 위반 행 제거 |

---
# 0. 파라미터 설정

In [ ]:
dbutils.widgets.text("run_date", "", "실행 날짜 (YYYY-MM-DD, 비어있으면 어제)")

from datetime import date, timedelta

run_date = dbutils.widgets.get("run_date")
if not run_date:
    run_date = str(date.today() - timedelta(days=1))

# 민감값은 Secret Scope에서 참조
BUCKET = dbutils.secrets.get(scope="kpay-poc", key="s3-bucket-name")

print(f"run_date : {run_date}")
print(f"S3 버킷  : {BUCKET}")

---
# 1. 카탈로그 / 스키마 준비

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS kpay_poc.silver;")
spark.sql("CREATE SCHEMA IF NOT EXISTS kpay_poc.gold;")
print("silver / gold 스키마 준비 완료")

---
# 2. Bronze 상태 확인

step1 적재 결과를 먼저 확인하고 진행.

In [ ]:
bronze_tables = [
    "kpay_poc.bronze.tiara_complete_log_raw",
    "kpay_poc.bronze.tiara_ns",
    "kpay_poc.bronze.payment_wide",
    "kpay_poc.bronze.an005d04",
    "kpay_poc.bronze.bp501d01",
]

print(f"{'테이블':<50} {'행 수':>15} {'rescued 행':>12}")
print("-" * 80)
for t in bronze_tables:
    try:
        cnt     = spark.sql(f"SELECT COUNT(*) as c FROM {t}").collect()[0]["c"]
        rescued = spark.sql(f"SELECT COUNT(*) as c FROM {t} WHERE _rescued_data IS NOT NULL").collect()[0]["c"]
        flag    = " ← 확인 필요" if rescued > 0 else ""
        print(f"{t:<50} {cnt:>15,} {rescued:>12,}{flag}")
    except Exception as e:
        print(f"{t:<50} 오류: {e}")

---
# 3. SDP 파이프라인 — 티아라

Bronze → Silver → Gold

> SDP 파일은 별도 Pipeline Job으로 등록 후 실행.  
> 아래 코드는 Pipeline 소스 파일에 붙여넣어 사용.

In [ ]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F

# ---------------------------------------------------------------------------
# Silver — 티아라 정제
# ---------------------------------------------------------------------------
@dp.table(
    name="silver_tiara",
    comment="티아라 행동로그 Silver — 품질 검증·타입 정규화·PII 마스킹 완료"
)
@dp.expect_all_or_drop({
    "pay_account_id 존재" : "pay_account_id IS NOT NULL",
    "event_time 유효"     : "event_time > '2020-01-01'",
    "action_type 존재"    : "action_type IS NOT NULL",
    "타입 미스매치 없음"  : "_rescued_data IS NULL",   # Bronze에서 rescued된 행 제외
})
def silver_tiara():
    return (
        dp.read_stream("kpay_poc.bronze.tiara_complete_log_raw")
            .select(
                F.col("pay_account_id"),
                F.col("event_time").cast("timestamp"),
                F.col("action_type"),
                F.col("service_name"),
                F.col("screen_name"),
                # PII 마스킹 — 전화번호 뒷자리 가림
                F.regexp_replace(F.col("phone_number"), r"(\d{3}-\d{4}-)(\d{4})", "$1****")
                  .alias("phone_number_masked"),
                F.to_date(F.col("event_time")).alias("event_date"),
            )
    )


# ---------------------------------------------------------------------------
# Gold — 서비스별 일별 행동 건수
# ---------------------------------------------------------------------------
@dp.materialized_view(
    name="gold_tiara_daily",
    comment="티아라 Gold — 서비스별 일별 행동 건수"
)
def gold_tiara_daily():
    return (
        dp.read("silver_tiara")
            .groupBy("event_date", "service_name", "action_type")
            .agg(
                F.count("*").alias("action_count"),
                F.countDistinct("pay_account_id").alias("unique_users"),
            )
    )

---
# 4. SDP 파이프라인 — 결제

In [ ]:
# ---------------------------------------------------------------------------
# Silver — 결제 Wide Table 정제
# 270컬럼, 600건/초 · 10KB/건
# ---------------------------------------------------------------------------
@dp.table(
    name="silver_payment",
    comment="결제 Wide Table Silver — 품질 검증·표준화 완료"
)
@dp.expect_all_or_drop({
    "payment_id 존재"    : "payment_id IS NOT NULL",
    "amount 양수"        : "amount > 0",
    "created_date 유효" : "created_date >= '2020-01-01'",
    "타입 미스매치 없음" : "_rescued_data IS NULL",
})
def silver_payment():
    return (
        dp.read_stream("kpay_poc.bronze.payment_wide")
            .select(
                F.col("payment_id"),
                F.col("pay_account_id"),
                F.col("amount").cast("decimal(20,2)"),
                F.col("currency"),
                F.col("payment_method"),
                F.col("status"),
                F.col("created_date").cast("date"),
                F.col("created_at").cast("timestamp"),
                F.col("merchant_id"),
                F.col("merchant_name"),
            )
    )


# ---------------------------------------------------------------------------
# Gold — 일별 결제 집계
# ---------------------------------------------------------------------------
@dp.materialized_view(
    name="gold_payment_daily",
    comment="결제 Gold — 일별·결제수단별 GMV 집계"
)
def gold_payment_daily():
    return (
        dp.read("silver_payment")
            .where(F.col("status") == "SUCCESS")
            .groupBy("created_date", "payment_method", "currency")
            .agg(
                F.sum("amount").alias("gmv"),
                F.count("payment_id").alias("tx_count"),
                F.countDistinct("pay_account_id").alias("unique_payers"),
                F.avg("amount").alias("avg_amount"),
            )
    )

---
# 5. Gold — 멱등 재실행 (replaceWhere)

> SDP 외부에서 Gold를 직접 PySpark로 생성할 때 사용.  
> 같은 `run_date`로 몇 번을 재실행해도 결과가 동일함 (멱등성 보장).

In [ ]:
def build_gold_tiara_daily(run_date: str) -> None:
    """
    Silver 티아라에서 특정 날짜의 Gold 집계를 생성.
    replaceWhere로 해당 날짜 파티션만 덮어써 멱등 재실행 보장.
    """
    src = (
        spark.table("kpay_poc.silver.silver_tiara")
             .where(F.col("event_date") == F.lit(run_date))
    )

    agg = (
        src.groupBy("event_date", "service_name", "action_type")
           .agg(
               F.count("*").alias("action_count"),
               F.countDistinct("pay_account_id").alias("unique_users"),
           )
    )

    (
        agg.write
           .format("delta")
           .mode("overwrite")
           .option("replaceWhere", f"event_date = '{run_date}'")   # 해당 날짜만 교체
           .saveAsTable("kpay_poc.gold.gold_tiara_daily")
    )
    print(f"[완료] gold_tiara_daily — run_date={run_date}")


def build_gold_payment_daily(run_date: str) -> None:
    """
    Silver 결제에서 특정 날짜의 Gold 집계를 생성.
    """
    src = (
        spark.table("kpay_poc.silver.silver_payment")
             .where(
                 (F.col("created_date") == F.lit(run_date)) &
                 (F.col("status") == "SUCCESS")
             )
    )

    agg = (
        src.groupBy("created_date", "payment_method", "currency")
           .agg(
               F.sum("amount").alias("gmv"),
               F.count("payment_id").alias("tx_count"),
               F.countDistinct("pay_account_id").alias("unique_payers"),
               F.avg("amount").alias("avg_amount"),
           )
    )

    (
        agg.write
           .format("delta")
           .mode("overwrite")
           .option("replaceWhere", f"created_date = '{run_date}'")  # 해당 날짜만 교체
           .saveAsTable("kpay_poc.gold.gold_payment_daily")
    )
    print(f"[완료] gold_payment_daily — run_date={run_date}")


print("Gold 빌더 함수 로드 완료")

In [ ]:
# Gold 실행 (run_date 기준 멱등 재실행 가능)
build_gold_tiara_daily(run_date)
build_gold_payment_daily(run_date)

---
# 6. Expectation 위반 현황 확인

SDP 파이프라인 실행 후 각 계층의 품질 지표를 확인.

In [ ]:
def quality_report(bronze_table, silver_table, label):
    print(f"\n=== [{label}] 품질 리포트 ===")

    bronze_cnt = spark.sql(f"SELECT COUNT(*) as c FROM {bronze_table}").collect()[0]["c"]
    silver_cnt = spark.sql(f"SELECT COUNT(*) as c FROM {silver_table}").collect()[0]["c"]
    rescued    = spark.sql(
        f"SELECT COUNT(*) as c FROM {bronze_table} WHERE _rescued_data IS NOT NULL"
    ).collect()[0]["c"]

    drop_rate    = (bronze_cnt - silver_cnt) / bronze_cnt * 100 if bronze_cnt > 0 else 0
    rescued_rate = rescued / bronze_cnt * 100 if bronze_cnt > 0 else 0

    print(f"  Bronze 행 수     : {bronze_cnt:>15,}")
    print(f"  Silver 행 수     : {silver_cnt:>15,}")
    print(f"  Expectation 제외 : {bronze_cnt - silver_cnt:>15,}  ({drop_rate:.2f}%)")
    print(f"  타입 미스매치    : {rescued:>15,}  ({rescued_rate:.2f}%)")

    if drop_rate > 5:
        print(f"  [경고] 제외율 {drop_rate:.1f}% — Expectation 조건 재검토 필요")


quality_report(
    "kpay_poc.bronze.tiara_complete_log_raw",
    "kpay_poc.silver.silver_tiara",
    "티아라"
)
quality_report(
    "kpay_poc.bronze.payment_wide",
    "kpay_poc.silver.silver_payment",
    "결제"
)

---
# 7. Gold 결과 확인

In [ ]:
# 티아라 Gold — 서비스별 상위 10
print(f"=== gold_tiara_daily ({run_date}) ===")
spark.sql(f"""
    SELECT service_name, action_type, action_count, unique_users
    FROM kpay_poc.gold.gold_tiara_daily
    WHERE event_date = '{run_date}'
    ORDER BY action_count DESC
    LIMIT 10
""").show(truncate=False)

In [ ]:
# 결제 Gold — 결제수단별 GMV
print(f"=== gold_payment_daily ({run_date}) ===")
spark.sql(f"""
    SELECT payment_method, currency,
           gmv, tx_count, unique_payers,
           ROUND(avg_amount, 0) as avg_amount
    FROM kpay_poc.gold.gold_payment_daily
    WHERE created_date = '{run_date}'
    ORDER BY gmv DESC
""").show(truncate=False)

---
# 8. Time Travel — 이전 버전 비교

Delta Lake Time Travel로 Gold 테이블의 이전 버전과 현재를 비교.

In [ ]:
# 현재 버전
current = spark.sql(f"""
    SELECT SUM(gmv) as total_gmv, SUM(tx_count) as total_tx
    FROM kpay_poc.gold.gold_payment_daily
    WHERE created_date = '{run_date}'
""").collect()[0]

# 이전 버전 (VERSION AS OF 0 = 최초 적재 시점)
try:
    prev = spark.sql(f"""
        SELECT SUM(gmv) as total_gmv, SUM(tx_count) as total_tx
        FROM kpay_poc.gold.gold_payment_daily VERSION AS OF 0
        WHERE created_date = '{run_date}'
    """).collect()[0]

    print(f"=== Time Travel 비교 ({run_date}) ===")
    print(f"  현재 GMV  : {current['total_gmv']:>20,.0f}")
    print(f"  이전 GMV  : {prev['total_gmv']:>20,.0f}")
    print(f"  현재 건수 : {current['total_tx']:>20,}")
    print(f"  이전 건수 : {prev['total_tx']:>20,}")
except Exception as e:
    print(f"이전 버전 없음 (최초 실행): {e}")

---
# 9. 전체 파이프라인 요약

In [ ]:
summary = [
    ("kpay_poc.bronze.tiara_complete_log_raw", "Bronze"),
    ("kpay_poc.bronze.payment_wide",           "Bronze"),
    ("kpay_poc.silver.silver_tiara",           "Silver"),
    ("kpay_poc.silver.silver_payment",         "Silver"),
    ("kpay_poc.gold.gold_tiara_daily",         "Gold"),
    ("kpay_poc.gold.gold_payment_daily",       "Gold"),
]

print(f"{'계층':<8} {'테이블':<50} {'행 수':>15}")
print("-" * 78)
for t, layer in summary:
    try:
        cnt = spark.sql(f"SELECT COUNT(*) as c FROM {t}").collect()[0]["c"]
        print(f"{layer:<8} {t:<50} {cnt:>15,}")
    except Exception as e:
        print(f"{layer:<8} {t:<50} {'오류':>15}")

print(f"\nrun_date: {run_date} — 파이프라인 완료")